# Stage 2 — Temporal Windowing (tracks.csv -> windows.json)

Pure Python/pandas. No model, no GPU, no `ultralytics` dependency -- which means, unlike
Stage 1a/1b, every function here can be executed end-to-end against synthetic data and
actually verified before it ever touches real tracking output.

**Input:** `tracks_all.csv` from Stage 1b (`video_id, camera_id, frame_id, timestamp_sec, x1, y1, x2, y2, confidence, class_id, class_name, track_id`).

**Output:** `windows.json` -- one record per (video, sliding time window), each listing the tracks present with dwell time, displacement, zone, proximity, carrying-object association, and rule-derived candidate events (enter / exit / loiter / stationary / place / pickup).


## 0. Setup

In [1]:
import os, json, math
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

print("Setup ready.")


Setup ready.


## 1. Config

`TRACKS_CSV` should point at the `tracks_all.csv` Stage 1b produced (single-video or combined multi-video -- this stage handles both, since everything is grouped by `video_id`).


In [2]:
TRACKS_CSV   = "/kaggle/input/datasets/aryanakhare/tracks/tracks_all.csv"      # <-- Stage 1b output
OUTPUT_PATH  = "/kaggle/working/windows.json"

WINDOW_SEC = 10.0     # sliding window length
STRIDE_SEC = 5.0      # hop between window starts (overlap = WINDOW_SEC - STRIDE_SEC)

# Thresholds are expressed in "person-heights" (see Section 2), not raw pixels, so they
# stay meaningful whether the camera is a wide rooftop VIRAT shot or a close-range clip.
STATIONARY_SPEED_PH_PER_SEC = 0.15   # person-heights/sec below this counts as "stationary"
LOITER_DWELL_SEC            = 8.0    # dwell threshold for a loiter candidate
LOITER_MAX_DISPLACEMENT_PH  = 1.5    # and net displacement must stay below this
PROXIMITY_THRESHOLD_PH      = 1.2    # "near" = centroids within this many person-heights

# How tightly a Carrying_object's box must sit against a Person's box to count as "held".
# Deliberately loose to start -- see the note in Section 8 about tightening this once you
# see real footage.
HELD_PROXIMITY_SCALE = 1.0

# Calibration-free default: split each frame into a 3x3 grid and name the cells. Leaves a
# path open (ZONES_YAML) to swap in real polygon zones later without touching the rest of
# the pipeline -- see Section 3.
ZONES_YAML = None   # e.g. "/kaggle/working/zones.yaml" once you've calibrated real polygons

print(f"Window: {WINDOW_SEC}s, stride: {STRIDE_SEC}s, overlap: {WINDOW_SEC - STRIDE_SEC}s")


Window: 10.0s, stride: 5.0s, overlap: 5.0s


## 2. Load tracks and compute per-video normalization

Frame size and a per-video "person-height" unit are derived here. Using the **median** Person bbox height (not mean/max) keeps the normalization robust to a few bad boxes.

In [3]:
def load_tracks(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"video_id", "camera_id", "frame_id", "timestamp_sec",
                "x1", "y1", "x2", "y2", "confidence", "class_id",
                "class_name", "track_id"}
    missing = required - set(df.columns)
    assert not missing, f"tracks.csv missing columns: {missing}"
    df["cx"] = (df.x1 + df.x2) / 2.0
    df["cy"] = (df.y1 + df.y2) / 2.0
    df["bbox_h"] = df.y2 - df.y1
    return df.sort_values(["video_id", "track_id", "timestamp_sec"]).reset_index(drop=True)


def person_height_units(df: pd.DataFrame) -> dict:
    """Median Person bbox height per video_id, in pixels. Falls back to the median
    height of ALL boxes in that video if no Person track exists (keeps this from
    crashing on a clip with no people, e.g. a vehicle-only test video)."""
    units = {}
    for vid, g in df.groupby("video_id"):
        persons = g[g.class_name == "Person"]
        if len(persons):
            units[vid] = float(persons.bbox_h.median())
        else:
            units[vid] = float(g.bbox_h.median()) if len(g) else 1.0
    return units


if os.path.exists(TRACKS_CSV):
    tracks_df = load_tracks(TRACKS_CSV)
    PH_UNITS = person_height_units(tracks_df)
    print(f"Loaded {len(tracks_df)} rows, {tracks_df.video_id.nunique()} video(s), "
          f"{tracks_df.track_id.nunique()} track_id values (not globally unique across videos)")
    print("Person-height units (px) per video:", PH_UNITS)
else:
    print(f"NOTE: {TRACKS_CSV} not found yet -- run the synthetic self-test in Section 6 "
          f"first, then come back and point this at real Stage 1b output.")
    tracks_df, PH_UNITS = None, None


Loaded 334 rows, 1 video(s), 11 track_id values (not globally unique across videos)
Person-height units (px) per video: {'Burglary001_x264A': 148.6}


## 3. Zones

Default: a zero-calibration 3x3 grid, so the pipeline runs immediately on any video without per-camera setup. `ZONES_YAML` is a hook for real polygon zones once you've pulled a reference frame and drawn them -- swap in a real point-in-polygon lookup there without changing anything downstream.

In [4]:
GRID_NAMES = [
    "top_left",    "top_mid",    "top_right",
    "mid_left",    "center",     "mid_right",
    "bottom_left", "bottom_mid", "bottom_right",
]

def grid_zone(cx: float, cy: float, frame_w: float, frame_h: float) -> str:
    col = min(int(cx / frame_w * 3), 2)
    row = min(int(cy / frame_h * 3), 2)
    return GRID_NAMES[row * 3 + col]


def frame_bounds(df: pd.DataFrame) -> dict:
    """Approximate frame size per video from the max observed box extent, since Stage 1b
    doesn't currently carry frame width/height through to tracks.csv."""
    bounds = {}
    for vid, g in df.groupby("video_id"):
        bounds[vid] = (float(g.x2.max()), float(g.y2.max()))
    return bounds


def load_zone_lookup(zones_yaml):
    """Returns a callable zone_fn(cx, cy, frame_w, frame_h) -> zone name.
    Falls back to the 3x3 grid when no calibrated zones file is given."""
    if zones_yaml is None or not os.path.exists(zones_yaml):
        return grid_zone
    # Real-polygon path: load zones_yaml (name -> polygon points) and do a
    # point-in-polygon test here instead. Left as a hook -- not needed for the MVP.
    raise NotImplementedError("Polygon zone loading not wired up yet -- add when you "
                              "calibrate ZONES_YAML against a real reference frame.")


ZONE_FN = load_zone_lookup(ZONES_YAML)
print(f"Using {'3x3 grid (calibration-free)' if ZONES_YAML is None else ZONES_YAML}")


Using 3x3 grid (calibration-free)


## 4. Per-track global stats

The fix for the enter/exit duplication bug lives here: every downstream comparison for "did this track start/end inside this window" is checked against these **global** (whole-video) first/last timestamps -- never against the window's own clipped slice.

In [5]:
def track_global_stats(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(["video_id", "track_id"]).agg(
        class_name=("class_name", "first"),
        t_start=("timestamp_sec", "min"),
        t_end=("timestamp_sec", "max"),
    ).reset_index()
    return g


def sliding_windows(t_min: float, t_max: float, window_sec: float, stride_sec: float):
    """Yield (start, end) window bounds covering [t_min, t_max]."""
    if t_max <= t_min:
        yield (t_min, t_min + window_sec)
        return
    start = t_min
    while start < t_max:
        yield (round(start, 3), round(start + window_sec, 3))
        start += stride_sec


## 5. Per-window feature computation

For every track present in a window: trajectory (centroid path, net displacement, mean speed), dwell, zone(s) touched, proximity to other tracks, and Carrying_object <-> Person association. Enter/exit/loiter/stationary/place candidates are attached per track here, compared against **global** track bounds (Section 4's fix).

In [6]:
def compute_window_features(win_df: pd.DataFrame, global_stats: pd.DataFrame,
                             video_id: str, w_start: float, w_end: float,
                             ph_unit: float, frame_w: float, frame_h: float) -> list:
    tracks_out = []
    gstats = global_stats[global_stats.video_id == video_id].set_index("track_id")

    per_track_centroids = {}   # track_id -> list of (t, cx, cy) for proximity pass below

    for track_id, tg in win_df.groupby("track_id"):
        tg = tg.sort_values("timestamp_sec")
        class_name = tg.class_name.iloc[0]

        t_first, t_last = float(tg.timestamp_sec.min()), float(tg.timestamp_sec.max())
        dwell_sec = t_last - t_first

        dx = tg.cx.diff().fillna(0.0)
        dy = tg.cy.diff().fillna(0.0)
        step_dist = np.sqrt(dx**2 + dy**2)
        path_len_px = float(step_dist.sum())
        displacement_px = float(math.hypot(tg.cx.iloc[-1] - tg.cx.iloc[0],
                                            tg.cy.iloc[-1] - tg.cy.iloc[0]))
        mean_speed_px_s = path_len_px / dwell_sec if dwell_sec > 0 else 0.0
        mean_speed_ph_s = mean_speed_px_s / ph_unit

        zones = sorted({ZONE_FN(row.cx, row.cy, frame_w, frame_h) for row in tg.itertuples()})

        # --- candidate events, checked against GLOBAL track bounds, not window bounds ---
        g = gstats.loc[track_id] if track_id in gstats.index else None
        candidates = []
        if g is not None:
            # A track only "enters" in the window that actually contains its global start,
            # and only "exits" in the window containing its global end. This is the fix for
            # the bug where every pass-through window fired both.
            if w_start <= g.t_start < w_end:
                candidates.append("enter")
            if w_start <= g.t_end < w_end:
                candidates.append("exit")

        if (dwell_sec >= LOITER_DWELL_SEC and
                displacement_px / ph_unit <= LOITER_MAX_DISPLACEMENT_PH):
            candidates.append("loiter")
        if mean_speed_ph_s <= STATIONARY_SPEED_PH_PER_SEC:
            candidates.append("stationary")

        per_track_centroids[track_id] = list(zip(tg.timestamp_sec, tg.cx, tg.cy))

        tracks_out.append({
            "track_id": int(track_id),
            "class": class_name,
            "dwell_sec": round(dwell_sec, 2),
            "displacement_px": round(displacement_px, 1),
            "mean_speed": round(mean_speed_px_s, 1),
            "zones": zones,
            "near": [],          # filled in by proximity pass below
            "candidate_events": candidates,
        })

    # --- proximity pass: pairwise, using overlapping timestamps only ---
    by_id = {t["track_id"]: t for t in tracks_out}
    for id_a, id_b in combinations(per_track_centroids.keys(), 2):
        pts_a = {round(t, 1): (cx, cy) for t, cx, cy in per_track_centroids[id_a]}
        pts_b = {round(t, 1): (cx, cy) for t, cx, cy in per_track_centroids[id_b]}
        shared_t = sorted(set(pts_a) & set(pts_b))
        if not shared_t:
            continue
        overlap_sec = 0.0
        near_any = False
        for t in shared_t:
            ax, ay = pts_a[t]; bx, by = pts_b[t]
            dist_ph = math.hypot(ax - bx, ay - by) / ph_unit
            if dist_ph <= PROXIMITY_THRESHOLD_PH:
                near_any = True
                overlap_sec += 0.1   # approx, since shared_t is at ~0.1s resolution here
        if near_any:
            by_id[id_a]["near"].append({"track_id": int(id_b),
                                          "class": by_id[id_b]["class"],
                                          "overlap_sec": round(overlap_sec, 1)})
            by_id[id_b]["near"].append({"track_id": int(id_a),
                                          "class": by_id[id_a]["class"],
                                          "overlap_sec": round(overlap_sec, 1)})

    return tracks_out


## 6. Carrying_object <-> Person association (held_by / released_at / place)

A Carrying_object box that stays within `HELD_PROXIMITY_SCALE` person-heights of a Person box's center is considered "held" by that person. When the held link ends and the object promptly goes stationary, that's a `place` candidate -- one of the events the architecture doc says should be derivable by rule, without the VLM.

In [7]:
def associate_carrying_objects(win_df: pd.DataFrame, tracks_out: list, ph_unit: float) -> None:
    """Mutates tracks_out in place, adding held_by / released_at to Carrying_object entries."""
    by_id = {t["track_id"]: t for t in tracks_out}
    persons = win_df[win_df.class_name == "Person"]
    objects = win_df[win_df.class_name == "Carrying_object"]
    if objects.empty or persons.empty:
        return

    threshold_px = HELD_PROXIMITY_SCALE * ph_unit

    for obj_id, og in objects.groupby("track_id"):
        og = og.sort_values("timestamp_sec")
        held_by, released_at = None, None
        currently_held = None

        for row in og.itertuples():
            same_t = persons[np.isclose(persons.timestamp_sec, row.timestamp_sec, atol=0.05)]
            nearest, nearest_dist = None, float("inf")
            for p in same_t.itertuples():
                d = math.hypot(row.cx - p.cx, row.cy - p.cy)
                if d < nearest_dist:
                    nearest, nearest_dist = p.track_id, d

            if nearest is not None and nearest_dist <= threshold_px:
                if held_by is None:
                    held_by = int(nearest)
                currently_held = True
            else:
                if currently_held:
                    released_at = float(row.timestamp_sec)
                currently_held = False

        if obj_id in by_id:
            if held_by is not None:
                by_id[obj_id]["held_by"] = held_by
            if released_at is not None:
                by_id[obj_id]["released_at"] = round(released_at, 2)
                if by_id[obj_id].get("mean_speed", 999) / ph_unit <= STATIONARY_SPEED_PH_PER_SEC:
                    by_id[obj_id]["candidate_events"].append("place")
            elif held_by is not None:
                by_id[obj_id]["candidate_events"].append("pickup")


## 7. Dedup pass

Because stride < window length, windows overlap -- so a timestamp near the boundary between two overlapping windows can independently satisfy both. Without this pass, the same enter/exit fires twice. This collapses exact (video, track_id, event_type, global timestamp) duplicates -- it does **not** collapse loiter/stationary entries that appear in multiple overlapping windows, since those are legitimate different sub-spans of the same ongoing behavior, not duplicate detections of a single instant.

In [8]:
def dedup_enter_exit(windows: list, global_stats: pd.DataFrame) -> list:
    seen = set()
    gstats = global_stats.set_index(["video_id", "track_id"])

    for w in windows:
        vid = w["video_id"]
        for t in w["tracks"]:
            if "enter" in t["candidate_events"] or "exit" in t["candidate_events"]:
                key_time = float(gstats.loc[(vid, t["track_id"])].t_start) \
                    if "enter" in t["candidate_events"] else None
                new_events = []
                for ev in t["candidate_events"]:
                    if ev not in ("enter", "exit"):
                        new_events.append(ev)
                        continue
                    ts = (gstats.loc[(vid, t["track_id"])].t_start if ev == "enter"
                          else gstats.loc[(vid, t["track_id"])].t_end)
                    dedup_key = (vid, t["track_id"], ev, round(float(ts), 1))
                    if dedup_key not in seen:
                        seen.add(dedup_key)
                        new_events.append(ev)
                t["candidate_events"] = new_events
    return windows


## 8. Build windows.json for one or more videos

Runs Sections 4-7 across every `video_id` present in the loaded tracks. Casts every value to native Python types before `json.dump` -- `pandas.groupby().min()/.max()` return `numpy.int64`/`float64`, which `json.dump` silently cannot serialize; this only surfaces as a crash at export time, not when you eyeball values with `print()`.

In [9]:
def to_native(obj):
    """Recursively cast numpy scalars to native Python types so json.dump doesn't choke."""
    if isinstance(obj, dict):
        return {k: to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_native(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def build_windows(df: pd.DataFrame, ph_units: dict, window_sec: float, stride_sec: float) -> list:
    global_stats = track_global_stats(df)
    bounds = frame_bounds(df)
    all_windows = []

    for video_id, vdf in df.groupby("video_id"):
        camera_id = vdf.camera_id.iloc[0]
        ph_unit = ph_units.get(video_id, 1.0)
        frame_w, frame_h = bounds[video_id]
        t_min, t_max = float(vdf.timestamp_sec.min()), float(vdf.timestamp_sec.max())

        for w_start, w_end in sliding_windows(t_min, t_max, window_sec, stride_sec):
            win_df = vdf[(vdf.timestamp_sec >= w_start) & (vdf.timestamp_sec < w_end)]
            if win_df.empty:
                continue
            tracks_out = compute_window_features(win_df, global_stats, video_id,
                                                  w_start, w_end, ph_unit, frame_w, frame_h)
            associate_carrying_objects(win_df, tracks_out, ph_unit)
            all_windows.append({
                "video_id": video_id,
                "camera_id": camera_id,
                "window": [round(w_start, 2), round(w_end, 2)],
                "tracks": tracks_out,
            })

    all_windows = dedup_enter_exit(all_windows, global_stats)
    return to_native(all_windows)


if tracks_df is not None:
    windows = build_windows(tracks_df, PH_UNITS, WINDOW_SEC, STRIDE_SEC)
    with open(OUTPUT_PATH, "w") as f:
        json.dump({"windows": windows}, f, indent=2)   # will raise if any numpy type leaked
    print(f"Saved {len(windows)} windows -> {OUTPUT_PATH}")
else:
    print("Skipped -- run Section 6's self-test below, or point TRACKS_CSV at real "
          "Stage 1b output and re-run from Section 2.")


Saved 10 windows -> /kaggle/working/windows.json


## 9. Self-test on synthetic data

Runs the full pipeline against a synthetic two-track clip (one Person, one Carrying_object) and checks the three things that broke in earlier versions: enter/exit fire exactly once per track across 5 overlapping windows, the numpy-type JSON export doesn't crash, and a place event is actually derived when the object is released and goes stationary.

In [10]:
def make_synthetic_tracks() -> pd.DataFrame:
    rows = []
    video_id, camera_id = "synthetic_test", "synthetic_test"

    # Person (track 1): walks left to right over 0-20s, carrying an object that gets
    # placed down and released around t=12s.
    for t in np.arange(0, 20.0, 0.5):
        rows.append({"video_id": video_id, "camera_id": camera_id,
                     "frame_id": int(t * 25), "timestamp_sec": round(t, 2),
                     "x1": 100 + t * 15, "y1": 200, "x2": 140 + t * 15, "y2": 300,
                     "confidence": 0.9, "class_id": 0, "class_name": "Person", "track_id": 1})

    # Carrying_object (track 2): glued to the person until t=12, then stays put (placed).
    for t in np.arange(0, 12.0, 0.5):
        rows.append({"video_id": video_id, "camera_id": camera_id,
                     "frame_id": int(t * 25), "timestamp_sec": round(t, 2),
                     "x1": 105 + t * 15, "y1": 220, "x2": 125 + t * 15, "y2": 260,
                     "confidence": 0.8, "class_id": 1, "class_name": "Carrying_object",
                     "track_id": 2})
    placed_x = 105 + 12 * 15
    for t in np.arange(12.0, 20.0, 0.5):
        rows.append({"video_id": video_id, "camera_id": camera_id,
                     "frame_id": int(t * 25), "timestamp_sec": round(t, 2),
                     "x1": placed_x, "y1": 220, "x2": placed_x + 20, "y2": 260,
                     "confidence": 0.8, "class_id": 1, "class_name": "Carrying_object",
                     "track_id": 2})

    return load_tracks_from_df(pd.DataFrame(rows))


def load_tracks_from_df(df: pd.DataFrame) -> pd.DataFrame:
    df["cx"] = (df.x1 + df.x2) / 2.0
    df["cy"] = (df.y1 + df.y2) / 2.0
    df["bbox_h"] = df.y2 - df.y1
    return df.sort_values(["video_id", "track_id", "timestamp_sec"]).reset_index(drop=True)


test_df = make_synthetic_tracks()
test_ph_units = person_height_units(test_df)
test_windows = build_windows(test_df, test_ph_units, WINDOW_SEC, STRIDE_SEC)

# json.dump must not raise -- this is the numpy-type regression test
json.dumps(test_windows)

enter_count = sum(1 for w in test_windows for t in w["tracks"]
                  if t["track_id"] == 1 and "enter" in t["candidate_events"])
exit_count = sum(1 for w in test_windows for t in w["tracks"]
                  if t["track_id"] == 1 and "exit" in t["candidate_events"])
place_count = sum(1 for w in test_windows for t in w["tracks"]
                   if t["track_id"] == 2 and "place" in t["candidate_events"])

print(f"Windows generated: {len(test_windows)}")
print(f"Track 1 (Person) enter fires: {enter_count} (expect 1)")
print(f"Track 1 (Person) exit fires:  {exit_count} (expect 1)")
print(f"Track 2 (Carrying_object) place fires: {place_count} (expect >= 1)")

assert enter_count == 1, "enter fired more than once -- dedup or global-bounds check broke"
assert exit_count == 1, "exit fired more than once -- dedup or global-bounds check broke"
assert place_count >= 1, "place was never derived -- check HELD_PROXIMITY_SCALE / association"
print("\nSelf-test passed.")


Windows generated: 4
Track 1 (Person) enter fires: 1 (expect 1)
Track 1 (Person) exit fires:  1 (expect 1)
Track 2 (Carrying_object) place fires: 2 (expect >= 1)

Self-test passed.


## Notes

- **`HELD_PROXIMITY_SCALE = 1.0`** is deliberately loose so the association logic fires on a first real run rather than silently missing everything. If you see spurious `carry`/`place` events on actual Stage 1b output, tighten this toward **0.4-0.5** rather than assuming the logic itself is broken.
- **Zones default to the calibration-free 3x3 grid** (Section 3). Swap in real polygon zones later via `ZONES_YAML` once you've pulled a reference frame and calibrated -- nothing else in the notebook needs to change.
- **`windows.json`'s `candidate_events`** per track is exactly what Stage 4 (Qwen2.5-VL) will be asked to confirm, reject, or correct -- this stage is not trying to be perfect, it's trying to give the VLM a strong prior instead of an unconstrained "what's happening?" prompt.
